# Geospatial Visualization: State Parks, F1 Circuits, and US Housing Inventory

This project is a tour through geospatial visualization in Python. I plot Indiana state parks as markers on an interactive folium map, add popups and a connecting route between them, map Formula 1 circuit locations worldwide, then merge Zillow's state-level housing inventory data with US state boundary shapefiles (geopandas) to build a choropleth map colored by average housing inventory.

## Mapping Indiana State Parks with Folium

For this project we need all these libraries 

In [94]:
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm

#1.1 - We make the locationsDF to hold the latitude and longitude values

In [95]:
locationsDF = pd.DataFrame({
    "latitude": [
        40.4973294, 39.8840311, 41.3347403, 38.7703088, 39.2980325,
        39.9294967, 39.1933212, 41.5350929, 38.7317623, 39.8683896
    ],
    "longitude": [
        -86.8525167, -87.239568, -85.3986565, -85.4466076, -86.7343327,
        -87.0827666, -86.2268856, -86.3719797, -86.4207084, -86.0219136
    ],
    "Park Name": [
        "Prophetstown", "Turkey Run", "Chain O'Lakes", "Clifty Falls", "McCormick's Creek",
        "Shades", "Brown County", "Potato Creek", "Spring Mill", "Fort Harrison"
    ]
})

In [96]:
points = gpd.GeoDataFrame(locationsDF, geometry=gpd.points_from_xy(locationsDF.longitude, locationsDF.latitude), crs="EPSG:4326")

We can use folium to make a blank map of the continents 

#1.2 - We can now make a map using folium 

In [97]:
my_map = folium.Map(location=[39.77, -86.29], zoom_start=6)
for index, row in points.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2
    ).add_to(my_map)

#1.3 - And then we can add circles around the state parks so it is easier to see them 

In [ ]:
my_map

(See screenshot in README)

## Adding Markers, Popups, and Connecting Routes

#2.1 - We can also add a pin icon on the state parks 

In [99]:
pin_map = folium.Map(location=[39.77, -86.29], zoom_start=6)
for index, row in points.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        radius=2
    ).add_to(pin_map)
pin_map

#2.2 - We can also add pop ups to display the correct park name 

In [ ]:
name_map = folium.Map(location=[39.77, -86.29], zoom_start=6)
for index, row in points.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=row['Park Name'],
        radius=2
    ).add_to(name_map)
name_map

(See screenshot in README)

In [101]:
coords = list(zip(locationsDF['latitude'], locationsDF['longitude']))
coords

[(40.4973294, -86.8525167),
 (39.8840311, -87.239568),
 (41.3347403, -85.3986565),
 (38.7703088, -85.4466076),
 (39.2980325, -86.7343327),
 (39.9294967, -87.0827666),
 (39.1933212, -86.2268856),
 (41.5350929, -86.3719797),
 (38.7317623, -86.4207084),
 (39.8683896, -86.0219136)]

#2.3 - We can add a line to connect all the points on the map 

In [102]:
line_map = folium.Map(location=[39.77, -86.29], zoom_start=6)
for index, row in points.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=row['Park Name'],
        radius=2
    ).add_to(line_map)

In [ ]:
folium.PolyLine(locations=coords, color="red", weight=4).add_to(line_map)
line_map

(See screenshot in README)

## Plotting Formula 1 Circuit Locations

We can read in the f1 data set and examin it 

In [104]:
f1 = pd.read_csv('/anvil/projects/tdm/data/formula_1/circuits.csv')

In [105]:
f1.shape

(77, 9)

In [106]:
f1.head()

,circuitId,circuitRef,name,location,country,lat,lng,alt,url
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800,10,http://en.wikipedia.org/wiki/Melbourne_Grand_P...
1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800,18,http://en.wikipedia.org/wiki/Sepang_Internatio...
2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060,7,http://en.wikipedia.org/wiki/Bahrain_Internati...
3,4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57000,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcel...
4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500,130,http://en.wikipedia.org/wiki/Istanbul_Park


In [107]:
testDF = pd.DataFrame({"Latitude": f1['lat'], "Longitude": f1['lng']})

#3.1 - We can look at the head of testDF to check the columns are correct 

In [108]:
testDF.head()

,Latitude,Longitude
0,-37.84970,144.96800
1,2.76083,101.73800
2,26.03250,50.51060
3,41.57000,2.26111
4,40.95170,29.40500


#3.2 - We can make points on the map for the f1 tracks 

In [109]:
my_map = folium.Map()

In [ ]:
for index, row in testDF.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5
    ).add_to(my_map)
my_map

(See screenshot in README)

#3.3 - And then we can focus in on 1 track so it zooms in automatically

In [112]:
my_map = folium.Map(location=[25.4885597,51.4477457], zoom_start=15)

In [113]:
my_map

## Merging Zillow Housing Data with State Boundaries

We can read in the two new data sets: Zillow state time and US TIGER

In [114]:
myDF = pd.read_csv('/anvil/projects/tdm/data/zillow/State_time_series.csv')
states_sf = gpd.read_file('/anvil/projects/tdm/data/tiger/state/tl_2025_us_state.shp')

#4.1 - We can clean up the DF first before working with it 

In [115]:
myDF_cleaned = myDF.dropna(subset=["RegionName"]).copy()
myDF_cleaned['RegionName'] = myDF_cleaned['RegionName'].str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)

In [116]:
myDF_cleaned.shape

(13212, 82)

In [117]:
myDF_cleaned['RegionName'].value_counts()

RegionName
Alabama                261
Arizona                261
Arkansas               261
California             261
Colorado               261
Connecticut            261
Delaware               261
Florida                261
Georgia                261
Hawaii                 261
Idaho                  261
Illinois               261
Indiana                261
Iowa                   261
Kansas                 261
Kentucky               261
Louisiana              261
Maine                  261
Maryland               261
Massachusetts          261
Michigan               261
Minnesota              261
Mississippi            261
Missouri               261
Nebraska               261
Nevada                 261
New Hampshire          261
New Jersey             261
New Mexico             261
New York               261
North Carolina         261
Ohio                   261
Oklahoma               261
Oregon                 261
Pennsylvania           261
Rhode Island           261
South Carolina   

#4.2 - We can look at the coord data in GPS format for states_sf

In [118]:
states_sf.crs

<Geographic 2D CRS: EPSG:4269>
Name: NAD83
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. Puerto Rico. United States (USA) - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Hawaii; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming. US Virgin Islands. British Virgin Islands

In [119]:
states_sf = states_sf.to_crs(epsg=4326)
states_sf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

We can group y state to get the avg InventoryRaw_AllHomes value per state

In [120]:
myDF_avg = myDF_cleaned.groupby('RegionName', as_index = False)['InventoryRaw_AllHomes'].mean()

In [121]:
myDF_avg.shape

(52, 2)

In [122]:
states_sf.shape

(56, 16)

#4.3 - With this command we can join the zillow data with states_sf. We can look at the shape and head to confirm it worked 

In [123]:
us50_states = states_sf.merge(myDF_avg, left_on='NAME', right_on='RegionName', how='left')

In [124]:
us50_states = us50_states.dropna(subset=['RegionName'])

In [125]:
us50_states.shape

(50, 18)

In [126]:
us50_states.head()

,REGION,DIVISION,STATEFP,STATENS,GEOID,GEOIDFQ,STUSPS,NAME,LSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,RegionName,InventoryRaw_AllHomes
0,1,2,36,01779796,36,0400000US36,NY,New York,00,G4000,A,122049344560,19256566831,+42.9133974,-075.5962723,"MULTIPOLYGON (((-74.72623 44.99486, -74.72257 ...",New York,87310.968750
1,4,8,32,01779793,32,0400000US32,NV,Nevada,00,G4000,A,284537074263,1839852286,+39.3310928,-116.6151469,"POLYGON ((-119.32418 41.99392, -119.32362 41.9...",Nevada,18081.958333
2,4,9,02,01785533,02,0400000US02,AK,Alaska,00,G4000,A,1479893380150,244326118163,+63.3473560,-152.8397334,"MULTIPOLYGON (((-167.55823 60.22436, -167.5567...",Alaska,3608.437500
4,1,1,50,01779802,50,0400000US50,VT,Vermont,00,G4000,A,23872664356,1030573104,+44.0589536,-072.6710173,"POLYGON ((-72.04187 44.15665, -72.0418 44.1566...",Vermont,6733.791667
5,1,1,09,01779780,09,0400000US09,CT,Connecticut,00,G4000,A,12542101087,1816013585,+41.5798637,-072.7466572,"POLYGON ((-72.5279 41.17777, -72.55156 41.1732...",Connecticut,24715.833333


## Choropleth Map of Housing Inventory by State

#5.1 - We can start with an empty mapping space 

In [127]:
my_map = folium.Map(location=[37, -95], zoom_start=3, tiles=None)

In [128]:
my_palette = cm.LinearColormap(colors=["Red", "Blue"],
                               vmin=us50_states['InventoryRaw_AllHomes'].min(),
                               vmax=us50_states['InventoryRaw_AllHomes'].max()
                              )

In [129]:
folium.GeoJson(
    us50_states,
    style_function=lambda feature: {
        'fillColor': my_palette(
            feature['properties']['InventoryRaw_AllHomes']
        ),

        # state outline settings
        # color, lineweight, color opacity level (0-1)
        'color': "Black",
        'weight': 1,
        'fillOpacity': 1
    },

    tooltip=folium.GeoJsonTooltip(
      # look at the states and inventory counts columns of us50_states
        fields=['RegionName', 'InventoryRaw_AllHomes'],
      # the popup will display these respective labels
        aliases=['State:', 'Inventory:'],
        localize=True
    )
).add_to(my_map)

In [ ]:
my_palette.caption = "Zillow hosuing"
my_palette.add_to(my_map)

(See screenshot in README)

In [ ]:
my_map

(See screenshot in README)